In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from scipy.stats import bootstrap

In [10]:
def load_data(path):
    return pd.read_csv(path)

def prepare_data(data):
    data.dropna(inplace = True)
    features = data.drop(['product', 'id'], axis = 1)
    target = data['product']
    return train_test_split(features, target, test_size = 0.25, random_state = 12345)

In [11]:
def train_and_evaluate(X_train, X_valid, y_train, y_valid):
    model = LinearRegression()
    model.fit(X_train, y_train)
    predictions = model.predict(X_valid)
    rmse = mean_squared_error(y_valid, predictions, squared=False)
    mean_predicted = predictions.mean()
    
    return model, predictions, rmse, mean_predicted

In [12]:
def calculate_revenue(predictions, targets, count=200):
    selected_indices = predictions.argsort()[-count:]
    selected_targets = targets.iloc[selected_indices]
    total_product = selected_targets.sum()
    revenue = total_product * 4500
    return revenue

In [28]:
def bootstrap_profit(predictions, targets, n_iterations = 1000, sample_size = 200, budget = 100_000_000):
    profits = []
    np.random.seed(12345)
    
    for _ in range(n_iterations):
        indices = np.random.choice(len(predictions), size=sample_size, replace=True)
        sample_targets = targets.iloc[indices]
        revenue = sample_targets.sum() * 4500
        profit = revenue - budget
        profits.append(profit)
    
    profits = np.array(profits)
    mean_profit = profits.mean()
    conf_int = np.percentile(profits, [2.5, 97.5])
    loss_risk = (profits < 0).mean() * 100
    
    return mean_profit, conf_int, loss_risk

In [29]:
def process_region(path):
    data = load_data(path)
    X_train, X_valid, y_train, y_valid = prepare_data(data)
    model, predictions, rmse, mean_predicted = train_and_evaluate(X_train, X_valid, y_train, y_valid)

    print(f"Modelo entrenado para {path}")
    print(f"RMSE: {rmse:.2f}")
    print(f"Predicción media: {mean_predicted:.2f}")
    print()

    predictions_valid = model.predict(X_valid)
    predictions_valid = pd.Series(predictions_valid)
    y_valid = y_valid.reset_index(drop=True)
    
    revenue = calculate_revenue(predictions_valid, y_valid)
    mean_profit, conf_int, loss_risk = bootstrap_profit(predictions_valid, y_valid)

    return {
        'path': path,
        'model': model,
        'rmse': rmse,
        'mean_prediction': mean_predicted,
        'revenue': revenue,
        'mean_profit': mean_profit,
        'conf_int': conf_int,
        'loss_risk': loss_risk
    }

In [30]:
regions = ['/datasets/geo_data_0.csv', '/datasets/geo_data_1.csv', '/datasets/geo_data_2.csv']
results = []

for region in regions:
    result = process_region(region)
    results.append(result)

# Mostrar resumen
for i, r in enumerate(results):
    print(f"Región {i}")
    print(f"RMSE: {r['rmse']:.2f}")
    print(f"Media predicha: {r['mean_prediction']:.2f}")
    print(f"Ingreso estimado (top 200): ${r['revenue']:,.2f}")
    print(f"Ganancia media (bootstrap): ${r['mean_profit']:,.2f}")
    print(f"Intervalo de confianza 95%: {r['conf_int']}")
    print(f"Riesgo de pérdida: {r['loss_risk']:.2f}%")
    print("-" * 40)

Modelo entrenado para /datasets/geo_data_0.csv
RMSE: 37.58
Predicción media: 92.59

Modelo entrenado para /datasets/geo_data_1.csv
RMSE: 0.89
Predicción media: 68.73

Modelo entrenado para /datasets/geo_data_2.csv
RMSE: 40.03
Predicción media: 94.97

Región 0
RMSE: 37.58
Media predicha: 92.59
Ingreso estimado (top 200): $133,208,260.43
Ganancia media (bootstrap): $-17,023,035.68
Intervalo de confianza 95%: [-22150501.85368042 -11716958.76809722]
Riesgo de pérdida: 100.00%
----------------------------------------
Región 1
RMSE: 0.89
Media predicha: 68.73
Ingreso estimado (top 200): $124,150,866.97
Ganancia media (bootstrap): $-38,175,035.91
Intervalo de confianza 95%: [-44153428.85774526 -32351554.76612842]
Riesgo de pérdida: 100.00%
----------------------------------------
Región 2
RMSE: 40.03
Media predicha: 94.97
Ingreso estimado (top 200): $127,103,499.64
Ganancia media (bootstrap): $-14,477,411.43
Intervalo de confianza 95%: [-20130888.5316936   -9032576.68765059]
Riesgo de pérdida

In [16]:
valid_regions = [r for r in results if r['loss_risk'] < 2.5]

if valid_regions:
    best_region = max(valid_regions, key = lambda r: r['mean_profit'])
    print(f"La mejor región para invertir es: {best_region['path']}")
else:
    print("No hay ninguna región que cumpla con el criterio de riesgo.")

No hay ninguna región que cumpla con el criterio de riesgo.


# Conclusión 

Región 1 tiene el modelo más preciso, pero el terreno tiene poco crudo, por lo tanto no sirve para el negocio.

Regiones 0 y 2 tienen un volumen ligeramente mayor, pero los modelos tienen mucha incertidumbre y predicciones poco confiables.

El hecho de que ninguna predicción promedio supere las 111.1 mil unidades por pozo ya es una alerta clara de que el proyecto no será rentable en ninguna región, sin importar la precisión del modelo.

No se recomienda realizar inversiones en ninguna de las tres regiones analizadas.

Todos los escenarios evaluados conducen a pérdidas significativas, incluso en los mejores casos posibles según el intervalo de confianza del 95%.

Aunque la predicción técnica en algunas regiones fue razonable (como en geo_data_1), el volumen promedio estimado no cumple con el umbral mínimo de rentabilidad (111.1 mil barriles por pozo).
